In [92]:
!pip install lightfm -qqq

In [93]:
from lightfm import LightFM
from lightfm.data import Dataset as LFMDataset
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import svds
from tqdm import tqdm

## <div style="padding: 30px;color:white;margin:10;font-size:60%;text-ali_gn:left;display:fill;border-radius:10px;background-color:#FFFFFF;overflow:hidden;background-color:#3b3745"><b><span style='color:#686dec'>1 |</span></b> <b>Введение</b></div>

#### <b><span style='color:#686dec'>LightFM</span></b>

`LightFM` это гибридный алгоритм рекомендаций, который сочетает в себе методы `коллаборативной фильтрации` и контентной фильтрации. Он особенно подходит для сценариев с разреженными данными, например, когда взаимодействия пользователей с объектами ограничены.

## <div style="padding: 30px;color:white;margin:10;font-size:60%;text-ali_gn:left;display:fill;border-radius:10px;background-color:#FFFFFF;overflow:hidden;background-color:#3b3745"><b><span style='color:#686dec'>2 |</span></b> <b>Данные</b></div>

In [94]:
df = pd.read_csv('/kaggle/input/mtc-data/mts_lib.csv')
df.head()

,user_id,item_id,progress,rating,start_date
0,126706,14433,80,NaN,2018-01-01
1,127290,140952,58,NaN,2018-01-01
2,66991,198453,89,NaN,2018-01-01
3,46791,83486,23,5.0,2018-01-01
4,79313,188770,88,5.0,2018-01-01


In [95]:
def get_data_info(data, 
                  user_id='user_id', 
                  item_id='item_id'):
    print(f'Размер датасета = {data.shape[0]}, \nколичество пользователей = {data[user_id].nunique()}, \nколичество объектов = {data[item_id].nunique()}')

In [96]:
# конвертируем в дату
df.loc[:, 'start_date'] = pd.to_datetime(df['start_date'], 
                                         format="%Y-%m-%d")

# удаляем дубликаты, оставляя последний по времени
df = df.sort_values('start_date').drop_duplicates(subset=['user_id', 'item_id'], 
                                                  keep='last')

get_data_info(df)

Размер датасета = 1532998, 
количество пользователей = 151600, 
количество объектов = 59599


## <div style="padding: 30px;color:white;margin:10;font-size:60%;text-ali_gn:left;display:fill;border-radius:10px;background-color:#FFFFFF;overflow:hidden;background-color:#3b3745"><b><span style='color:#686dec'>1 |</span></b> <b>Предобработка</b></div>

#### <b><span style='color:#686dec'>Фильтрация</span></b>

У нас дв

In [97]:
df = df[df['progress'] > 30]

In [98]:
def filter_data(df, user_count=5, item_count=5):
    
    item_counts = df.groupby('item_id')['user_id'].count()
    pop_items = item_counts[item_counts >= user_count]
    df_implicit = df[df['item_id'].isin(pop_items.index)]

    user_counts = df.groupby('user_id')['item_id'].count()
    pop_users = user_counts[user_counts >= item_count]
    df = df[df['user_id'].isin(pop_users.index)].copy()
    return df

df = filter_data(df)
get_data_info(df)

Размер датасета = 654819, 
количество пользователей = 64955, 
количество объектов = 59485


In [99]:
u_features = pd.read_csv('/kaggle/input/mtc-data/users.csv')
i_features = pd.read_csv('/kaggle/input/mtc-data/items.csv')
i_features.rename(columns={'id': 'item_id'}, inplace=True)

In [100]:
# make sure the feature are present in rating dataframe df
i_features = i_features[i_features['item_id'].isin(df['item_id'])].copy()
u_features = u_features[u_features['user_id'].isin(df['user_id'])].copy()

In [101]:
user_idx = df['user_id'].astype('category').cat.codes
item_idx = df['item_id'].astype('category').cat.codes
user2id = dict(zip(df['user_id'], user_idx))
item2id = dict(zip(df['item_id'], item_idx))

In [102]:
# user features
u_features.head()

,user_id,age,sex
0,1,45_54,NaN
2,3,65_inf,0.0
10,11,55_64,0.0
11,12,55_64,1.0
12,13,25_34,0.0


In [103]:
# item features 
i_features.head()

,item_id,title,genres,authors,year
0,128115,Ворон-челобитчик,"Зарубежные детские книги,Сказки,Зарубежная кла...",Михаил Салтыков-Щедрин,1886
1,210979,Скрипка Ротшильда,"Классическая проза,Литература 19 века,Русская ...",Антон Чехов,1894
2,95632,Испорченные дети,"Зарубежная классика,Классическая проза,Литерат...",Михаил Салтыков-Щедрин,1869
3,247906,Странный человек,"Пьесы и драматургия,Литература 19 века",Михаил Лермонтов,1831
4,294280,Господа ташкентцы,"Зарубежная классика,Классическая проза,Литерат...",Михаил Салтыков-Щедрин,1873


In [104]:
u_features['age'].value_counts()

age
18_24     21396
25_34     12207
35_44      7422
55_64      7256
45_54      6186
65_inf     4016
Name: count, dtype: int64

### Train-Test Splitting

In [105]:
def train_test_split(X, user_col, time_col):
    full_history = X.sort_values([user_col, time_col]).groupby(user_col)
    test = full_history.tail(1)
    train = full_history.head(-1)
    return train, test

train, test = train_test_split(df, 'user_id', 'start_date')

In [106]:
# test contains the last user activity; progress/rating
test.head()

,user_id,item_id,progress,rating,start_date
1366165,0,193970,42,NaN,2019-10-14 00:00:00
874014,1,32364,100,NaN,2019-02-22 00:00:00
1514829,3,125376,88,NaN,2019-12-23 00:00:00
1276428,11,21283,40,NaN,2019-08-31 00:00:00
990325,12,77735,34,NaN,2019-04-18 00:00:00


### Light-FM

- `LightFM` объединяет коллабративную фильтрацию с контентными подходами.
- Эмбеддинги пользователей и эмбеддинги объектов можно обучать как с использованием их признаков, так и без.
- Если признаки используются, то эмбеддинги пользователей и объектов представляют собой сумму векторов их признаков (включая id как признак).

#### User Features

In [107]:
u_features.head()

,user_id,age,sex
0,1,45_54,NaN
2,3,65_inf,0.0
10,11,55_64,0.0
11,12,55_64,1.0
12,13,25_34,0.0


Create Feature Name & Value Array; `u_features_list`

In [108]:
# set index as user_id
u_features.set_index('user_id', inplace=True)

# create feature name & value 
u_features_list = u_features.apply(
    lambda feature_values: [f'{feature}_{feature_values[feature]}' 
                            for feature in feature_values.index 
                            if not pd.isna(feature_values[feature])],
    axis=1
    )
u_features_list = u_features_list.rename('features')
u_features_list

user_id
1                   [age_45_54]
3         [age_65_inf, sex_0.0]
11         [age_55_64, sex_0.0]
12         [age_55_64, sex_1.0]
13         [age_25_34, sex_0.0]
                  ...          
159603     [age_18_24, sex_1.0]
159605     [age_18_24, sex_0.0]
159606     [age_25_34, sex_0.0]
159607              [age_25_34]
159610     [age_35_44, sex_0.0]
Name: features, Length: 58529, dtype: object

In [109]:
# all unique feature name & value 
user_tags = set(u_features_list.explode().dropna().values)
user_tags

{'age_18_24',
 'age_25_34',
 'age_35_44',
 'age_45_54',
 'age_55_64',
 'age_65_inf',
 'sex_0.0',
 'sex_1.0'}

#### Item Features

Возмем топ 50 жанров

In [110]:
# book genre
i_features['genres']

0        Зарубежные детские книги,Сказки,Зарубежная кла...
1        Классическая проза,Литература 19 века,Русская ...
2        Зарубежная классика,Классическая проза,Литерат...
3                   Пьесы и драматургия,Литература 19 века
4        Зарубежная классика,Классическая проза,Литерат...
                               ...                        
59594                Политология,Книги по экономике,Газеты
59595                Политология,Книги по экономике,Газеты
59596                     Политология,Общая история,Газеты
59597                                   Журнальные издания
59598    Журнальные издания,Энциклопедии,Научная фантас...
Name: genres, Length: 59485, dtype: object

In [111]:
i_features_lfm = i_features.copy()
i_features_lfm.set_index('item_id', inplace=True)

# how many books were read
i_features_lfm['reads'] = df.groupby('item_id')['user_id'].count()

# genre features (list)
i_features_lfm['genres'] = i_features_lfm['genres'].str.lower().str.split(',')
i_features_lfm['genres'] = i_features_lfm['genres'].apply(lambda x: x if isinstance(x, list) else [])

i_features_lfm.head()

,title,genres,authors,year,reads
item_id,,,,,
128115,Ворон-челобитчик,"[зарубежные детские книги, сказки, зарубежная ...",Михаил Салтыков-Щедрин,1886,11
210979,Скрипка Ротшильда,"[классическая проза, литература 19 века, русск...",Антон Чехов,1894,87
95632,Испорченные дети,"[зарубежная классика, классическая проза, лите...",Михаил Салтыков-Щедрин,1869,5
247906,Странный человек,"[пьесы и драматургия, литература 19 века]",Михаил Лермонтов,1831,6
294280,Господа ташкентцы,"[зарубежная классика, классическая проза, лите...",Михаил Салтыков-Щедрин,1873,7


In [112]:
# genre based read count
genres_count = i_features_lfm[['genres','reads']].explode('genres').groupby('genres')['reads'].sum()
genres_count.sort_values(ascending=False)

genres
любовное фэнтези               72008
попаданцы                      52832
современные любовные романы    49248
современные детективы          46660
героическое фэнтези            40794
                               ...  
литература 7 класс                 3
экономическая статистика           2
воздушный транспорт                2
научно-практические журналы        2
математика 3 класс                 1
Name: reads, Length: 640, dtype: int64

In [113]:
item_tags = genres_count.sort_values(ascending=False)[:50].index
item_tags[:10]

Index(['любовное фэнтези', 'попаданцы', 'современные любовные романы',
       'современные детективы', 'героическое фэнтези',
       'современная русская литература', 'боевая фантастика',
       'зарубежные любовные романы', 'боевое фэнтези', 'эротические романы'],
      dtype='object', name='genres')

In [114]:
def filter_genres(genres_list, valid_genres=None):
    if not genres_list:
        return []
    return [genre for genre in genres_list if genre in valid_genres]

# filter genres
i_features_lfm['features'] = i_features_lfm['genres'].apply(filter_genres, 
                                                            valid_genres=set(item_tags))
i_features_list = i_features_lfm['features']
i_features_lfm.head()

,title,genres,authors,year,reads,features
item_id,,,,,,
128115,Ворон-челобитчик,"[зарубежные детские книги, сказки, зарубежная ...",Михаил Салтыков-Щедрин,1886,11,"[зарубежная классика, литература 19 века, русс..."
210979,Скрипка Ротшильда,"[классическая проза, литература 19 века, русск...",Антон Чехов,1894,87,"[литература 19 века, русская классика]"
95632,Испорченные дети,"[зарубежная классика, классическая проза, лите...",Михаил Салтыков-Щедрин,1869,5,"[зарубежная классика, литература 19 века, русс..."
247906,Странный человек,"[пьесы и драматургия, литература 19 века]",Михаил Лермонтов,1831,6,[литература 19 века]
294280,Господа ташкентцы,"[зарубежная классика, классическая проза, лите...",Михаил Салтыков-Щедрин,1873,7,"[зарубежная классика, литература 19 века, русс..."


In [115]:
lfm_dataset = LFMDataset()

In [116]:
print('user_id',df['user_id'].unique()[:5])
print('item_id',df['item_id'].unique()[:5])

print(user_tags,item_tags)

user_id [47427 99355 55263 58868 40184]
item_id [ 46915 249281  80651 164458 128111]
{'age_25_34', 'sex_1.0', 'age_65_inf', 'age_55_64', 'sex_0.0', 'age_45_54', 'age_35_44', 'age_18_24'} Index(['любовное фэнтези', 'попаданцы', 'современные любовные романы',
       'современные детективы', 'героическое фэнтези',
       'современная русская литература', 'боевая фантастика',
       'зарубежные любовные романы', 'боевое фэнтези', 'эротические романы',
       'книги про волшебников', 'остросюжетные любовные романы',
       'короткие любовные романы', 'магические академии',
       'зарубежные детективы', 'триллеры', 'научная фантастика',
       'русская классика', 'иронические детективы', 'юмористическое фэнтези',
       'саморазвитие / личностный рост', 'космическая фантастика', 'мистика',
       'литература 19 века', 'городское фэнтези', 'биографии и мемуары',
       'газеты', 'историческая фантастика', 'журнальные издания',
       'литература 20 века', 'современная зарубежная литература',

In [117]:
# set unique users & items 
lfm_dataset.fit_partial(users=df['user_id'].unique(),  
                        items=df['item_id'].unique())

# user features & item features
lfm_dataset.fit_partial(user_features=user_tags, 
                        item_features=item_tags)

# 
user_mapping, item_mapping = lfm_dataset.mapping()[0], lfm_dataset.mapping()[2]

In [118]:
inv_user_mapping = {value: key for key, value in user_mapping.items()}
inv_item_mapping = {value: key for key, value in item_mapping.items()}

print('user mapping')
for ii,(i,j) in enumerate(user_mapping.items()):
    if(ii<5):
        print(i,j)
        
print('\nitem mapping')
for ii,(i,j) in enumerate(item_mapping.items()):
    if(ii<5):
        print(i,j)

user mapping
47427 0
99355 1
55263 2
58868 3
40184 4

item mapping
46915 0
249281 1
80651 2
164458 3
128111 4


In [119]:
lfm_dataset.interactions_shape() # user, items matrix

(64955, 59485)

In [120]:
# Строем признаки пользователей и объектов
sparse_i_features = lfm_dataset.build_item_features([[row.item_id, row.features] for row in i_features_list.reset_index().itertuples()])
sparse_u_features = lfm_dataset.build_user_features([[row.user_id, row.features] for row in u_features_list.reset_index().itertuples()])

In [121]:
sparse_i_features

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 131534 stored elements and shape (59485, 59535)>

In [122]:
i_features_list[51]

['героическое фэнтези',
 'зарубежная фантастика',
 'зарубежное фэнтези',
 'попаданцы']

In [123]:
sparse_i_features[51, :].nonzero(), sparse_i_features[51, :].data

((array([0], dtype=int32), array([51], dtype=int32)),
 array([1.], dtype=float32))

In [124]:
(interactions, weights) = lfm_dataset.build_interactions([(row.user_id, row.item_id, row.progress) for row in train.itertuples()])

Посмотрим на содержание матриц interactions и weights: используя матрицу interactions можно легко перейти от рейтингов к бинарному фидбеку. Будем использовать weights.

In [125]:
print(interactions.shape)
print(interactions.data) # interaction
print(weights.data) # progress

(64955, 59485)
[1 1 1 ... 1 1 1]
[65. 78. 77. ... 85. 60. 33.]


### Train Model

In [126]:
%%time
lightfm = LightFM(no_components=50, 
                  loss='warp')
lightfm.fit(interactions, user_features=sparse_u_features, item_features=sparse_i_features, epochs=40, num_threads=8)

CPU times: user 3min 48s, sys: 202 ms, total: 3min 49s
Wall time: 1min 1s


### Prediction for one user

In [127]:
train[train.user_id == 2535].head()

,user_id,item_id,progress,rating,start_date
584253,2535,255947,71,NaN,2018-10-08 00:00:00
1070601,2535,4014,33,5.0,2019-05-26 00:00:00
1127244,2535,93029,63,NaN,2019-06-21 00:00:00
1367405,2535,28889,82,NaN,2019-10-14 00:00:00
1366424,2535,143447,99,5.0,2019-10-14 00:00:00


In [128]:
pred = lightfm.predict(user_ids=user_mapping[2535], 
                       item_ids=sorted(item_mapping.values()), 
                       user_features=sparse_u_features, 
                       item_features=sparse_i_features)
pred

array([-152.62328, -151.48076, -154.31828, ..., -154.59323, -155.17883,
       -153.88129], dtype=float32)

In [129]:
k = 50
ids = np.argpartition(pred, - k)[- k:]
rel = pred[ids]
res = pd.DataFrame(zip(ids, rel), columns=['item_id', 'relevance'])
res['item_id'] = res['item_id'].map(inv_item_mapping)

In [130]:
res.merge(i_features).head()

,item_id,relevance,title,genres,authors,year
0,158191,-147.915848,Ежевичная зима,Современная зарубежная литература,Сара Джио,2012
1,212014,-147.914429,Пока я жива,Современная зарубежная литература,Дженни Даунхэм,2007
2,144939,-147.908539,Обратная сила. Том 3. 1983–1997,Современная русская литература,Александра Маринина,2016
3,298473,-147.896317,Хроники Раздолбая,Современная русская литература,Павел Санаев,2013
4,156325,-147.884689,Акушер-ХА! (сборник),Современная русская литература,Татьяна Соломатина,2009


In [137]:
item_id_lfm = item_mapping[212014]
user_id_lfm = user_mapping[2535]

### Prediction for one user (manual)

Вернемся к формуле:

$$r_{ui} = <q_u , p_i> + b_u + b_i$$

$$q_u = \sum_{j \in f_u} e^U_j$$
$$ e^U_j = w^U_j  e_j$$
$$b_u = \sum_{j \in f_u} b^U_j$$
$$ b^U_j = w^U_j  b_j$$

<br>

где $q_u, p_i$ - вектора пользователя и объекта, являющиеся суммой векторов и их признаков
$b_u, b_i$ - смещения для признаков пользователя и объекта


In [139]:
u_biases, u_vectors = lightfm.get_user_representations()
u_vectors.shape, u_biases.shape

((64963, 50), (64963,))

In [142]:
i_biases, i_vectors = lightfm.get_item_representations()
i_vectors.shape, i_biases.shape

((59535, 50), (59535,))

In [143]:
user_vector = sparse_u_features[user_id_lfm] @ u_vectors
item_vector = sparse_i_features[item_id_lfm] @ i_vectors
rel_ours = (user_vector @ item_vector.T + sparse_u_features[user_id_lfm] @ u_biases + sparse_i_features[item_id_lfm] @ i_biases).ravel()[0]

In [144]:
rel_ours

-147.9144